# Jour 2 — Embeddings Evo2 + classifieur (le « teacher »)

**Evo2** est un grand modèle de fondation pour l'ADN (architecture StripedHyena2) entraîné
sur des génomes à travers l'arbre du vivant. Plutôt que de concevoir des caractéristiques à
la main (k-mers) ou de les apprendre à partir de zéro sur notre petit jeu de données (le
CNN), on peut demander à un modèle qui a déjà « lu » une énorme quantité d'ADN de
représenter nos fenêtres pour nous — c'est le même geste « embeddings en boîte noire » que
ESM/ProtBERT pour les protéines.

Nous accédons à Evo2 via l'**API NIM hébergée par NVIDIA** plutôt qu'en exécutant nous-mêmes
le modèle à 7 milliards de paramètres. Les organisateurs ont déjà appelé l'API sur chaque
fenêtre étiquetée issue de `01_kmer_and_cnn_baselines.ipynb` et sauvegardé les résultats —
c'est le plus grand levier de faisabilité de la semaine : vous n'attendez jamais un appel
API en direct.

In [ ]:
import sys
sys.path.append("src")

import torch
from embeddings import load_supervised_embeddings
from models.classifier_heads import MLPHead
from eval import evaluate_logits, count_params, measure_latency_torch
from viz import plot_embedding_space

EMB_DIR = "../../data/supervised/embeddings"

# TODO : chargez les embeddings pré-calculés train/val avec load_supervised_embeddings(EMB_DIR, ...)
X_train, y_train, ids_train = ...
X_val, y_val, ids_val = ...
print("embedding dim:", X_train.shape[1], "| train windows:", X_train.shape[0])

## Visualisons l'espace des embeddings avant d'entraîner quoi que ce soit

In [ ]:
# TODO : appelez plot_embedding_space sur les embeddings d'entraînement (method="pca")
# avec un titre du type "Evo2 embeddings, colored by coding/non-coding label"
...

## Entraînons le teacher : une petite tête MLP sur les embeddings gelés

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)

# TODO : instanciez MLPHead(d_in=X_train.shape[1]) et un optimiseur Adam (lr=1e-3)
teacher = ...
optimizer = ...

n = X_train_t.shape[0]
for epoch in range(20):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        # TODO : logits du teacher sur ce mini-lot, puis perte BCE-with-logits
        logits = ...
        loss = ...
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1}: loss={epoch_loss/n:.4f}")

In [ ]:
# TODO : évaluez le teacher sur validation (sans gradient), puis calculez metrics/params/latence
with torch.no_grad():
    val_logits = ...
teacher_metrics = ...
print("Evo2 embeddings + MLP teacher:", teacher_metrics)
print("params:", ..., "| latency (ms/sample):", ...)

### Point de contrôle

Comparez `teacher_metrics` au tableau `comparison` du Jour 1 — les embeddings Evo2
devraient donner un gain de précision visible, et le graphique PCA devrait déjà montrer
une certaine séparation des clusters par étiquette, même si les embeddings n'ont jamais
été affinés pour cette tâche.

Gardez `teacher`, `X_train_t`, `y_train_t` (ou rechargez-les) pour le prochain notebook —
les prédictions du teacher deviennent les cibles douces (*soft targets*) pour la
distillation.

Suite : `03_knowledge_distillation.ipynb`.

*Bloqué ? La version complète est dans `solution/02_evo2_embeddings_and_classifier.ipynb`.*